# Projeto de Integração de Dados — Análise de Desempenho no Futebol
---
### Fonte de Dados
*   **Portal**: [Open Football Data](https://openfootball.github.io/)
*   **Descrição**: Base de dados aberta e colaborativa contendo históricos de partidas, placares e eventos de futebol internacional.
---
### Pergunta de Negócio & Hipótese Analítica
> **Eficiência Emocional**: *Qual é o impacto real no desempenho e postura tática dos times logo após sofrerem um gol?*
#### O que esta análise busca responder no pipeline:
1. **Padrão de Reação**: Quanto tempo o time leva, em média, para finalizar ou marcar um gol após sofrer um?
2. **Efeito Dominó**: Sofrer um gol aumenta estatisticamente a chance de sofrer o segundo em um curto intervalo de tempo?
3. **Resiliência**: Qual a taxa de conversão de derrotas em empates ou viradas de placar após o primeiro gol sofrido?

# Camada Bronze: Ingestão de Dados
Repositório JSON (o que vamos consumir):


EuroCopa: https://github.com/openfootball/euro.json/tree/master


Copa do Mundo: https://github.com/openfootball/worldcup.json/tree/master 

---
### Arquitetura
*Este notebook foi refatorado para ser **Agnóstico de Plataforma**. Os caminhos de armazenamento e os comandos proprietários do Databricks foram parametrizados usando variáveis. Isso significa que o pipeline pode ser executado no Databricks, em um Spark Local ou na Nuvem alterando apenas uma variável de ambiente*


---
### Arquitetura Agnóstica de Plataforma (Multi-Cloud & Local)

Este pipeline foi desenvolvido seguindo as melhores práticas de **Agnosticismo de Infraestrutura**. O processamento dos dados em PySpark e a manipulação do Delta Lake estão isolados das particularidades do provedor de nuvem.

O notebook utiliza a variável `ENVIRONMENT` para determinar dinamicamente os caminhos e as regras de gravação:

*   **`local`**: Executa no Docker/Máquina Física, consumindo e gravando dados nas pastas relativas `./data/raw` e `./data/bronze`. Ideal para desenvolvimento e testes unitários com zero custo de nuvem.
*   **`databricks`**: Ativa os caminhos seguros do *Unity Catalog* (`/Volumes/workspace/...`) e delega a criação dos *schemas* (bancos de dados) dinamicamente via Databricks SQL.
*   **`aws` / `azure` / `gcp`** *(Expansão Futura)*: Bastaria criar novos blocos `if` no código para apontar os caminhos para cloud storages mantendo toda a regra de negócio do PySpark intacta.

Dessa forma, o projeto evita o *Vendor Lock-in*.
---


In [0]:
# --- CONFIGURAÇÃO AGNÓSTICA DO AMBIENTE ---
import os
from pyspark.sql import SparkSession

# 1. Inicializa o Spark (Agnóstico: Cria localmente se não existir, usa o do Databricks se existir)
spark = SparkSession.builder.appName("FootballPipeline_Bronze").getOrCreate()

# 2. Variável que define o ambiente ('databricks', 'local', 'aws')
# Troque os comentários abaixo para alternar os ambientes
# ENVIRONMENT = os.getenv("SPARK_ENV", "databricks")
ENVIRONMENT = os.getenv("SPARK_ENV", "databricks")

# 3. Parametrização de caminhos
if ENVIRONMENT == "databricks":
    PATH_RAW = "/tmp/football_raw"                  # Area temporaria no cluster
    PATH_BRONZE = "/Volumes/workspace/bronze/football_raw" # Volume Unity Catalog
    CATALOG_NAME = "workspace.bronze"
else:
    PATH_RAW = "./data/raw"                         # Pasta local para arquivos crus
    PATH_BRONZE = "./data/bronze"                   # Pasta local para tabelas Delta
    CATALOG_NAME = "bronze"                         # Metastore local padrão

print(f"Ambiente Configurado: {ENVIRONMENT}")
print(f"Caminho RAW: {PATH_RAW}")
print(f"Caminho BRONZE: {PATH_BRONZE}")

# 4. Criacao automatica dos Schemas (Substituindo Terraform em contas limitadas)
if ENVIRONMENT == "databricks":
    print("\nCriando os bancos de dados (Schemas) via Spark SQL...")
    spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze;")
    spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver;")
    spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold;")
    print("Schemas Bronze, Silver e Gold criados/validados com sucesso!")


Ambiente Configurado: databricks
Caminho RAW: /tmp/football_raw
Caminho BRONZE: /Volumes/workspace/bronze/football_raw

Criando os bancos de dados (Schemas) via Spark SQL...
Schemas Bronze, Silver e Gold criados/validados com sucesso!


In [0]:
#Importando bibliotecas
import requests #pede para baixar os arquivos do Git
from requests.adapters import HTTPAdapter # Seguracas de rede, se tentar baixar um arquivo demora muito ele faz o retry
from urllib3.util import Retry
import os #permite gerenciar diretorios
import zipfile #permite descompactar os arquivos
import io #permite manipular os arquivos
from pyspark.sql import functions as F #permite manipular os dados

In [0]:
#Criando function

def baixar_arquivo(url_zip, pasta_destino):
    """
        Faz o download de um arquivo .zip do Git, extrai e salva em uma pasta local 
    """
    print(f"Iniciando download do ZIP: {url_zip}")

    #Configuracao do retry
    session = requests.Session()
    retries = Retry(total=5, backoff_factor=0.1, status_forcelist=[502, 503, 504])
    session.mount('https://', HTTPAdapter(max_retries=retries))

    try:
        #Faz o download do arquivo
        resposta = session.get(url_zip, timeout = 30)
        resposta.raise_for_status()

        #Abre o zip e extrai na pasta de destino
        with zipfile.ZipFile(io.BytesIO(resposta.content)) as zip_ref:
            os.makedirs(pasta_destino, exist_ok=True)
            zip_ref.extractall(pasta_destino)

            print(f"Conteudo extraido em:{pasta_destino}")
            return True
    except Exception as e:
        print(f"Erro ao baixar/extrair o arquivo: {e}")
        return False

In [0]:
# Parâmetros de Ingestão 
# Aqui iremos realizar a primeira carga completa (Full Load)

# JSON: Placares e Resultados, incluindo Copa do Mundo e Eurocopa (com minutos de gols)
fontes_json = [
    {"repositorio": "worldcup.json", "url": "https://github.com/openfootball/worldcup.json/archive/refs/heads/master.zip"},
    {"repositorio": "euro.json", "url": "https://github.com/openfootball/euro.json/archive/refs/heads/master.zip"}
]

# Pastas locais no cluster
pasta_raw_json = f"{PATH_RAW}/json/"

print(f"Fonte JSON configurada com Sucesso : {len(fontes_json)}")
print("Parâmetros Finalizados!")

Fonte JSON configurada com Sucesso : 2
Parâmetros Finalizados!


In [0]:
# Executa o download de todos os Zips

print("Iniciando o Full Load")

# 1. Baixar todos JSON
print("Baixando Fontes JSON")
for fonte in fontes_json:
    pasta_destino = pasta_raw_json + fonte["repositorio"]
    sucesso = baixar_arquivo(fonte["url"], pasta_destino)
    if not sucesso:
        raise Exception(f"Falha ao baixar: {fonte['repositorio']}")

print("Full Load concluido com sucesso!")

Iniciando o Full Load
Baixando Fontes JSON
Iniciando download do ZIP: https://github.com/openfootball/worldcup.json/archive/refs/heads/master.zip
Conteudo extraido em:/tmp/football_raw/json/worldcup.json
Iniciando download do ZIP: https://github.com/openfootball/euro.json/archive/refs/heads/master.zip
Conteudo extraido em:/tmp/football_raw/json/euro.json
Full Load concluido com sucesso!


In [0]:
# Verificando os arquivos antes de passar para o Spark

import os 

def listar_arquivos(pasta, extensao):
    """
        Lista todos os arquivos de forma recursiva
    """
    arquivos = []
    for raiz, dirs, files in os.walk(pasta):
        for arquivo in files:
            if arquivo.endswith(extensao):
                arquivos.append(os.path.join(raiz, arquivo))
    return arquivos

# Validando arquivos JSON 
arquivos_json = listar_arquivos(pasta_raw_json, ".json")
print(f"Arquivos JSON: {len(arquivos_json)}")
for f in arquivos_json[:5]: # Mostra os 5 primeiros
    print(f"  {f}")
print("         ")


Arquivos JSON: 44
  /tmp/football_raw/json/worldcup.json/worldcup.json-master/1990/worldcup.json
  /tmp/football_raw/json/worldcup.json/worldcup.json-master/2022/worldcup.json
  /tmp/football_raw/json/worldcup.json/worldcup.json-master/2022/worldcup.groups.json
  /tmp/football_raw/json/worldcup.json/worldcup.json-master/2022/worldcup.stadiums.json
  /tmp/football_raw/json/worldcup.json/worldcup.json-master/1978/worldcup.json
         


In [0]:
# CAMADA BRONZE: SCHEMA JSON (Placares, Resultados e GOLS!)

from pyspark.sql.types import (
    StructType, StructField,
    StringType, ArrayType, IntegerType, BooleanType
)

#  Schema específico para ler os detalhes de cada gol
schema_goal = StructType([
    StructField("name", StringType(), True),
    StructField("minute", IntegerType(), True),
    StructField("offset", IntegerType(), True),
    StructField("penalty", BooleanType(), True),
    StructField("owngoal", BooleanType(), True)
])

# 1. Schema do JSON para as Copas 
schema_json = StructType([
    StructField("name", StringType(), True),       # Nome da competicao
    StructField("matches", ArrayType(              # Lista de partidas
        StructType([
            StructField("round", StringType(), True),  # Rodada
            StructField("date",  StringType(), True),  # Data 
            StructField("time",  StringType(), True),  # Horario 
            StructField("team1", StringType(), True),  # Time mandante
            StructField("team2", StringType(), True),  # Time visitante
            StructField("score", StructType([
            # ft = placar final    [mandante, visitante]
            # ht = placar intervalo [mandante, visitante]
                StructField("ft", ArrayType(IntegerType()), True),
                StructField("ht", ArrayType(IntegerType()), True),
            ]), True),
            
            # Lendo a linha do tempo dos gols de cada time
            StructField("goals1", ArrayType(schema_goal), True), 
            StructField("goals2", ArrayType(schema_goal), True)  
        ])
    ), True)
])


In [0]:
# 2. Cria o SCHEMA e o volume no catalogo
if ENVIRONMENT == "databricks":
    # No Databricks criamos o Database. Localmente, apenas salvamos os arquivos físicos.
    spark.sql(f"""
    CREATE DATABASE IF NOT EXISTS {CATALOG_NAME};
    """)

if ENVIRONMENT == "databricks":
    # Criação de Volume é exclusiva do Databricks Unity Catalog
    spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.football_raw;
    """)




In [0]:
#Validando
if ENVIRONMENT == "databricks":
    spark.sql(f"SHOW VOLUMES IN {CATALOG_NAME}").show(truncate=False)


+--------+------------+
|database|volume_name |
+--------+------------+
|bronze  |football_raw|
+--------+------------+



In [0]:
# 3. Copiando arquivos JSON para o Volume do Unity Catalog

import shutil

# Caminho do volume no Unity Catalog
volume_path = f"{PATH_BRONZE}/"

# Cria subpasta no volume apenas para JSON
volume_json_path = volume_path + "json/"

os.makedirs(volume_json_path, exist_ok=True)

print("Copiando arquivos JSON para o volume...")

# Copia todos os arquivos JSON
for arquivo_json in arquivos_json:
    # Extrai o caminho relativo
    caminho_relativo = arquivo_json.replace(pasta_raw_json, "")
    destino = os.path.join(volume_json_path, caminho_relativo)
    
    # Cria o diretório de destino se necessário
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    
    # Copia o arquivo
    shutil.copy2(arquivo_json, destino)

print(f"✓ {len(arquivos_json)} arquivos JSON copiados para {volume_json_path}")
print(f"\n✓ Todos os arquivos foram copiados para o volume {CATALOG_NAME}.football_raw")

Copiando arquivos JSON para o volume...
✓ 44 arquivos JSON copiados para /Volumes/workspace/bronze/football_raw/json/

✓ Todos os arquivos foram copiados para o volume workspace.bronze.football_raw


In [0]:
# 4. Lendo arquivos JSON do Volume e aplicando o schema

print("Lendo arquivos JSON do Volume...")

df_raw_json = (
    spark.read
    .schema(schema_json)
    .option("multiLine", "true")
    .json(f"{PATH_BRONZE}/json/*/*/*/*.json")
    .withColumn("source_file", F.col("_metadata.file_path"))
)

#print(f"Total de arquivos JSON lidos: {df_raw_json.select('source_file').distinct().count()}")
print(f"Total de competições: {df_raw_json.count()}")

# Preview dos dados
print("\nPreview dos dados:")
df_raw_json.select("name", "source_file").show(5, truncate=False)

Lendo arquivos JSON do Volume...
Total de competições: 138

Preview dos dados:
+----------------------+-------------------------------------------------------------------------------------------------------------+
|name                  |source_file                                                                                                  |
+----------------------+-------------------------------------------------------------------------------------------------------------+
|Czech Republic        |dbfs:/Volumes/workspace/bronze/football_raw/json/worldcup.json/worldcup.json-master/2026/worldcup.squads.json|
|Mexico                |dbfs:/Volumes/workspace/bronze/football_raw/json/worldcup.json/worldcup.json-master/2026/worldcup.squads.json|
|South Africa          |dbfs:/Volumes/workspace/bronze/football_raw/json/worldcup.json/worldcup.json-master/2026/worldcup.squads.json|
|South Korea           |dbfs:/Volumes/workspace/bronze/football_raw/json/worldcup.json/worldcup.json-master/202

In [0]:
# 5. Transformando e estruturando os dados das partidas

print("Transformando dados...")

df_bronze_matches = (
    df_raw_json
    
    # 5.1 Abre a lista de partidas: 1 linha por partida
    .select(
        F.col("name").alias("competition_name"),
        F.col("source_file"),
        F.explode("matches").alias("match")
    )
    
    # 5.2 Extrai colunas separadas
    .select(
        F.col("competition_name"),
        F.col("source_file"),
        F.col("match.round").alias("round"),
        F.col("match.date").cast("date").alias("match_date"),
        F.col("match.time").alias("match_time"),
        F.col("match.team1").alias("team_home"),
        F.col("match.team2").alias("team_away"),
        F.col("match.score.ft")[0].alias("score_ft_home"),
        F.col("match.score.ft")[1].alias("score_ft_away"),
        F.col("match.score.ht")[0].alias("score_ht_home"),
        F.col("match.score.ht")[1].alias("score_ht_away")
    )
    .withColumn("ingested_at", F.current_timestamp())
)

print(f"Total de partidas processadas: {df_bronze_matches.count()}")
print("\nSchema da tabela de partidas:")
df_bronze_matches.printSchema()
print("\nAmostra dos dados:")
df_bronze_matches.show(5, truncate=False)

Transformando dados...
Total de partidas processadas: 1302

Schema da tabela de partidas:
root
 |-- competition_name: string (nullable = true)
 |-- source_file: string (nullable = false)
 |-- round: string (nullable = true)
 |-- match_date: date (nullable = true)
 |-- match_time: string (nullable = true)
 |-- team_home: string (nullable = true)
 |-- team_away: string (nullable = true)
 |-- score_ft_home: integer (nullable = true)
 |-- score_ft_away: integer (nullable = true)
 |-- score_ht_home: integer (nullable = true)
 |-- score_ht_away: integer (nullable = true)
 |-- ingested_at: timestamp (nullable = false)


Amostra dos dados:
+----------------+------------------------------------------------------------------------------------------------------+-----------+----------+-----------+--------------+--------------+-------------+-------------+-------------+-------------+--------------------------+
|competition_name|source_file                                                             

In [0]:
# 6. Salvando como tabela Delta na camada Bronze

print(f"Criando tabela Delta: {CATALOG_NAME}.matches...")

if ENVIRONMENT == "databricks":
    # No Unity Catalog, usamos saveAsTable() para criar tabelas gerenciadas
    df_bronze_matches.write \
        .format("delta") \
        .mode("overwrite") \
        .option("mergeSchema", "true") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{CATALOG_NAME}.matches")
else:
    # Localmente, salvamos no diretório físico
    df_bronze_matches.write \
        .format("delta") \
        .mode("overwrite") \
        .option("mergeSchema", "true") \
        .option("overwriteSchema", "true") \
        .save(f"{PATH_BRONZE}/matches")

print(f"✓ Tabela {CATALOG_NAME}.matches criada com sucesso!")

# Validando a tabela criada
total_registros = spark.table(f'{CATALOG_NAME}.matches').count()
print(f"✓ Total de partidas na tabela: {total_registros:,}")

# Mostra algumas estatísticas
print("\n📊 Estatísticas da tabela:")
spark.sql(f"""
    SELECT 
        COUNT(DISTINCT competition_name) as total_competicoes,
        COUNT(DISTINCT team_home) as total_times,
        MIN(match_date) as primeira_partida,
        MAX(match_date) as ultima_partida
    FROM {CATALOG_NAME}.matches
""").show(truncate=False)

Criando tabela Delta: workspace.bronze.matches...
✓ Tabela workspace.bronze.matches criada com sucesso!
✓ Total de partidas na tabela: 1,302

📊 Estatísticas da tabela:
+-----------------+-----------+----------------+--------------+
|total_competicoes|total_times|primeira_partida|ultima_partida|
+-----------------+-----------+----------------+--------------+
|28               |169        |1930-07-13      |2028-07-09    |
+-----------------+-----------+----------------+--------------+



In [0]:
%sql
-- Consultando as 10 primeiras partidas da tabela matches

SELECT 
    competition_name,
    match_date,
    team_home,
    team_away,
    score_ft_home,
    score_ft_away,
    round
FROM workspace.bronze.matches
ORDER BY match_date ASC
LIMIT 10

competition_name,match_date,team_home,team_away,score_ft_home,score_ft_away,round
World Cup 1930,1930-07-13,France,Mexico,4,1,Matchday 1
World Cup 1930,1930-07-13,United States,Belgium,3,0,Matchday 1
World Cup 1930,1930-07-14,Yugoslavia,Brazil,2,1,Matchday 2
World Cup 1930,1930-07-14,Romania,Peru,3,1,Matchday 2
World Cup 1930,1930-07-15,Argentina,France,1,0,Matchday 3
World Cup 1930,1930-07-16,Chile,Mexico,3,0,Matchday 4
World Cup 1930,1930-07-17,United States,Paraguay,3,0,Matchday 5
World Cup 1930,1930-07-17,Yugoslavia,Bolivia,4,0,Matchday 5
World Cup 1930,1930-07-18,Uruguay,Peru,1,0,Matchday 6
World Cup 1930,1930-07-19,Argentina,Mexico,6,3,Matchday 7
